In [ ]:
import qiskit
import numpy as np
import matplotlib.pyplot as plt
import warnings

from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.synthesis import SuzukiTrotter, LieTrotter
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
#from qiskit.visualization import array_to_latex

warnings.filterwarnings("ignore")

In [3]:
##define the transverse-field Ising Hamiltonian
def get_hamiltonian(nqubits,J,h,alpha):
    #tuples containing the Pauli string, 
    #the qubit indices corresponding to the Pauli string,
    #the coefficient.
    ZZ_tuples=[("ZZ",[i,i+1],-J) for i in range(0,nqubits-1)]
    Z_tuples=[("Z",[i],-h*np.sin(alpha)) for i in range(0,nqubits-1)]
    X_tuples=[("X",[i],-h*np.cos(alpha)) for i in range(0,nqubits-1)]
    hamiltonian=SparsePauliOp.from_sparse_list(
        [*ZZ_tuples,*Z_tuples,*X_tuples],num_qubits=nqubits)
    return hamiltonian.simplify()

In [13]:
#sample hamiltonian N=6, J=0.2, h=1.2, alpha=pi/8
n_qubits=6
hamiltonian=get_hamiltonian(n_qubits,J=0.2,h=1.2,alpha=np.pi/8.0)
#hamiltonian

In [14]:
#set the parameters of the time-evolution simulation
#Lie-Trotter, 2nd order Suzuki-Trotter, and 4th order Suzuki-Trotter
num_time_steps=60
evolution_time=30.0
dt=evolution_time/num_time_steps
product_formula_lt=LieTrotter()
product_formula_st2=SuzukiTrotter(order=2)
product_formula_st4=SuzukiTrotter(order=4)

In [19]:
#prepare the quantum circuit 1
initial_circuit=QuantumCircuit(n_qubits)
initial_circuit.prepare_state("001100")
#initial_circuit.decompose(reps=1).draw("mpl")